# Phase 0.2 — Feature Engineering & Pre-Computation

This notebook loads the cleaned data from `phase0_data_sampling.ipynb` and pre-computes
all features that can be built **before** any model is trained. The outputs are feature
matrices consumed by Phase 1 (NCF), Phase 2 (Content-Based), and Phase 3 (Wide & Deep).

---

### Notebook outline

| Section | What it does |
|---|---|
| **1. Temporal Train/Test Split** | Per-user 80/20 split on `timestamp_created` (at least 1 test review per user) |
| **2. Static Game Features** | Genre, category, platform multi-hot encodings + scalar features (price, metacritic, is_free, required_age, publisher portfolio size) assembled into a single game feature matrix |
| **3. Leakage-Safe LOO Aggregates** | User-level (positive ratio, avg playtime, price preference), game-level (positive ratio, avg playtime), and developer-level (avg rating) aggregates — all computed with leave-one-out exclusion to prevent label leakage |
| **4. Domain-Informed Negative Sampling** | Genre-proportional, platform-compatible, popularity-weighted synthetic negatives (4:1 ratio) via two-stage rejection sampling |
| **5. BERT Description Embeddings** | `all-MiniLM-L6-v2` encoding of `short_description`, PCA-reduced to 128 dims |
| **6. User Genre Preference Vector** | Playtime-weighted genre distribution from positively reviewed games (28-dim) |
| **7. Save** | All feature matrices saved as parquet |

### Feature computation pipeline

```
Pre-compute (this notebook)    → Phase 1 (NCF)       → Phase 2 (Content)     → Phase 3 (Wide & Deep)  → Phase 4 (Hybrid)
├─ Train/test split                    │                     │                        │                       │
├─ Static game metadata                │                     │                        │                       │
│  (genre/category multi-hot,          │                     │                        │                       │
│   price, metacritic, platform,       │                     │                        │                       │
│   is_free, publisher portfolio)      │                     │                        │                       │
├─ LOO aggregate features              │                     │                        │                       │
│  (user_positive_ratio,               ▼                     ▼                        ▼                       │
│   game_positive_ratio,         NCF user/item        Content-based            W&D predicted           Meta-learner
│   user_avg_playtime,           embeddings (32d)     similarity scores        probability             combines all
│   game_avg_playtime,                 │              (user taste profiles     (used by Phase 4)        phase scores
│   user_price_preference,       NCF predicted         computed in Phase 2)          │                       │
│   developer_avg_rating,         scores ─────────────────────────────────────────────────────────────►      │
│   playtime_vs_user_avg)              │                     │                        │                      ▼
├─ BERT description embeddings         │                     └────────────────────────────────────────► Final ranking
├─ User genre preference vectors       │                                              │
├─ Negative samples + features         └───── embeddings fed as features ─────────────┘
```

### What CAN be pre-computed (this notebook)

| Feature | Why it's safe to pre-compute |
|---|---|
| Train/test temporal split | Fixed once, used by everything |
| Static game metadata (genre, category, price, metacritic, platform) | Independent of any model or interaction |
| BERT embeddings of `short_description` | Editorial content, independent of user experience |
| LOO aggregates (user, game, developer, price) for training rows | Fixed per-row: exclude the specific (u,g) pair |
| LOO aggregates for test rows | Use only train data — fixed once split is made |
| Negative samples + their features | Generated from train set; no (u,g) pair to exclude |
| User genre preference vectors | Playtime-weighted genre multi-hot of positively reviewed games |

### What is computed in later phases

| Feature | Computed in | Why it can't be pre-computed | Consumed by |
|---|---|---|---|
| User taste profiles from review text | Phase 2 | Only needed for content-based similarity scoring | Phase 2 |
| NCF user embeddings (32-dim) | Phase 1 | Output of Phase 1 model | Phase 3 |
| NCF item embeddings (32-dim) | Phase 1 | Output of Phase 1 model | Phase 3 |
| NCF predicted scores | Phase 1 | Output of Phase 1 model | Phase 4 |
| Content-based similarity scores | Phase 2 | Output of Phase 2 computation | Phase 4 |

### Negative sample LOO nuance

For synthetic negative samples (user u, game g) where no real interaction exists:
- **User aggregates**: Use ALL of user u's training reviews (no exclusion needed — there's no real interaction to leak)
- **Game aggregates**: Use ALL of game g's training reviews (same reasoning)

## 0. Setup — Load Cleaned Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = "/content/drive/MyDrive/processed_steam_data"
OUTPUT_DIR = "/content/drive/MyDrive/feature_engineered_data"

reviews = pd.read_parquet(os.path.join(INPUT_DIR, "reviews_filtered.parquet"))
apps = pd.read_parquet(os.path.join(INPUT_DIR, "apps_filtered.parquet"))
app_genres = pd.read_parquet(os.path.join(INPUT_DIR, "app_genres_filtered.parquet"))
app_categories = pd.read_parquet(os.path.join(INPUT_DIR, "app_categories_filtered.parquet"))
app_developers = pd.read_parquet(os.path.join(INPUT_DIR, "app_developers_filtered.parquet"))
app_publishers = pd.read_parquet(os.path.join(INPUT_DIR, "app_publishers_filtered.parquet"))
app_platforms = pd.read_parquet(os.path.join(INPUT_DIR, "app_platforms_filtered.parquet"))
platforms = pd.read_parquet(os.path.join(INPUT_DIR, "platforms.parquet"))
developers = pd.read_parquet(os.path.join(INPUT_DIR, "developers.parquet"))
publishers = pd.read_parquet(os.path.join(INPUT_DIR, "publishers.parquet"))

print(f"Loaded: {len(reviews):,} reviews, {len(apps):,} apps")
print(f"Users: {reviews['author_steamid'].nunique():,}, Games: {reviews['appid'].nunique():,}")

Loaded: 224,895 reviews, 32,883 apps
Users: 36,840, Games: 32,883


## 1. Temporal Train/Test Split

Per-user split: for each user, the oldest 80% of reviews go to train, the most recent 20% go to test
(with at least 1 held-out review per user).

In [ ]:
# Sort by user and timestamp
reviews = reviews.sort_values(["author_steamid", "timestamp_created"]).reset_index(drop=True)

# Per-user split: last review per user goes to test, rest to train
# For users with >= 5 reviews, hold out the last 20%; for users with < 5, hold out the last 1
def assign_split(group):
    n = len(group)
    n_test = max(1, int(n * 0.2))  # at least 1 test sample
    split = ['train'] * (n - n_test) + ['test'] * n_test
    group['split'] = split
    return group

reviews = reviews.groupby("author_steamid", group_keys=False).apply(assign_split)

train_df = reviews[reviews["split"] == "train"].copy()
test_df = reviews[reviews["split"] == "test"].copy()

print(f"Train: {len(train_df):,} reviews ({len(train_df)/len(reviews)*100:.1f}%)")
print(f"Test:  {len(test_df):,} reviews ({len(test_df)/len(reviews)*100:.1f}%)")
print(f"\nTrain users: {train_df['author_steamid'].nunique():,}")
print(f"Test users:  {test_df['author_steamid'].nunique():,}")
print(f"\nTrain label dist: {train_df['voted_up'].value_counts().to_dict()}")
print(f"Test label dist:  {test_df['voted_up'].value_counts().to_dict()}")

/tmp/ipykernel_8062/145167388.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews = reviews.groupby("author_steamid", group_keys=False).apply(assign_split)


Train: 173,832 reviews (77.3%)
Test:  51,063 reviews (22.7%)

Train users: 36,161
Test users:  36,840

Train label dist: {True: 129846, False: 43986}
Test label dist:  {True: 38866, False: 12197}


## 2. Static Game Features

These are independent of any user interaction — safe to compute once.

### 2a. Genre Multi-Hot Encoding

In [ ]:
# Genre multi-hot: one column per canonical English genre
genre_pivot = app_genres.assign(val=1).pivot_table(
    index="appid", columns="genre_name", values="val", fill_value=0
)
genre_pivot.columns = [f"genre_{c}" for c in genre_pivot.columns]

print(f"Genre multi-hot: {genre_pivot.shape[0]:,} games x {genre_pivot.shape[1]} genres")
print(f"Games with genre info: {genre_pivot.shape[0]:,} / {len(apps):,}")
genre_pivot.head()

Genre multi-hot: 32,810 games x 28 genres
Games with genre info: 32,810 / 32,883


,genre_Accounting,genre_Action,genre_Adventure,genre_Animation & Modeling,genre_Audio Production,genre_Casual,genre_Design & Illustration,genre_Early Access,genre_Education,genre_Free To Play,...,genre_Racing,genre_Sexual Content,genre_Simulation,genre_Software Training,genre_Sports,genre_Strategy,genre_Utilities,genre_Video Production,genre_Violent,genre_Web Publishing
appid,,,,,,,,,,,,,,,,,,,,,
400,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1300,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1313,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1520,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


### 2b. Category Multi-Hot Encoding

In [ ]:
# Category multi-hot
cat_pivot = app_categories.assign(val=1).pivot_table(
    index="appid", columns="category_name", values="val", fill_value=0
)
cat_pivot.columns = [f"cat_{c}" for c in cat_pivot.columns]

print(f"Category multi-hot: {cat_pivot.shape[0]:,} games x {cat_pivot.shape[1]} categories")
cat_pivot.head()

Category multi-hot: 32,717 games x 58 categories


,cat_Adjustable Difficulty,cat_Adjustable Text Size,cat_Camera Comfort,cat_Captions available,cat_Chat Speech-to-text,cat_Chat Text-to-speech,cat_Co-op,cat_Color Alternatives,cat_Commentary available,cat_Cross-Platform Multiplayer,...,cat_SteamVR Collectibles,cat_Stereo Sound,cat_Subtitle Options,cat_Surround Sound,cat_Touch Only Option,cat_Tracked Controller Support,cat_VR Only,cat_VR Support,cat_VR Supported,cat_Valve Anti-Cheat enabled
appid,,,,,,,,,,,,,,,,,,,,,
400,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1520,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1640,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2c. Platform Multi-Hot Encoding

In [ ]:
# Platform multi-hot
platform_map = platforms.set_index("id")["name"].to_dict()
app_platforms_named = app_platforms.copy()
app_platforms_named["platform_name"] = app_platforms_named["platform_id"].map(platform_map)

plat_pivot = app_platforms_named.assign(val=1).pivot_table(
    index="appid", columns="platform_name", values="val", fill_value=0
)
plat_pivot.columns = [f"platform_{c}" for c in plat_pivot.columns]

print(f"Platform multi-hot: {plat_pivot.shape[0]:,} games x {plat_pivot.shape[1]} platforms")
plat_pivot.head()

Platform multi-hot: 32,883 games x 3 platforms


,platform_linux,platform_mac,platform_windows
appid,,,
400,1.0,0.0,1.0
1300,0.0,0.0,1.0
1313,0.0,0.0,1.0
1520,1.0,1.0,1.0
1640,0.0,0.0,1.0


### 2d. Scalar Game Features

In [ ]:
# Scalar features from apps table
game_scalar = apps[["appid", "is_free", "required_age", "metacritic_score",
                     "mat_final_price_log"]].copy()

# Impute metacritic_score with genre-median
# First get primary genre per game (most common genre assignment)
primary_genre = app_genres.drop_duplicates(subset="appid", keep="first").set_index("appid")["genre_name"]
game_scalar = game_scalar.merge(primary_genre.rename("primary_genre"), left_on="appid", right_index=True, how="left")

genre_median_metacritic = game_scalar.groupby("primary_genre")["metacritic_score"].transform("median")
game_scalar["metacritic_score"] = game_scalar["metacritic_score"].fillna(genre_median_metacritic)
# Fill remaining NaN with global median
global_median = game_scalar["metacritic_score"].median()
game_scalar["metacritic_score"] = game_scalar["metacritic_score"].fillna(global_median)

# Fill missing price with 0 (likely free games)
game_scalar["mat_final_price_log"] = game_scalar["mat_final_price_log"].fillna(0)

# is_free to int
game_scalar["is_free"] = game_scalar["is_free"].astype(int)

# Drop helper column
game_scalar = game_scalar.drop(columns=["primary_genre"])

print(f"Scalar game features: {game_scalar.shape}")
print(f"Missing values:\n{game_scalar.isnull().sum().to_string()}")
game_scalar.head()

Scalar game features: (32883, 5)
Missing values:
appid                  0
is_free                0
required_age           0
metacritic_score       0
mat_final_price_log    0


,appid,is_free,required_age,metacritic_score,mat_final_price_log
0,400,0,0,90.0,6.907755
1,1300,0,0,75.0,6.907755
2,1313,0,0,77.0,6.907755
3,1520,0,0,84.0,7.090077
4,1640,0,0,84.0,6.396930


### 2e. Publisher Portfolio Size

In [ ]:
# Publisher portfolio size (how many games does this game's publisher have?)
pub_portfolio = app_publishers.groupby("publisher_id")["appid"].nunique().rename("publisher_portfolio_size")
game_publisher = app_publishers.drop_duplicates(subset="appid", keep="first")  # primary publisher
game_publisher = game_publisher.merge(pub_portfolio, left_on="publisher_id", right_index=True, how="left")
game_publisher = game_publisher[["appid", "publisher_portfolio_size"]]

print(f"Publisher portfolio: {len(game_publisher):,} games")
print(game_publisher["publisher_portfolio_size"].describe().to_string())

Publisher portfolio: 32,648 games
count    32648.000000
mean        14.748622
std         34.550378
min          1.000000
25%          1.000000
50%          2.000000
75%         10.000000
max        263.000000


### 2f. Assemble Static Game Feature Matrix

In [ ]:
# Join all static game features
game_features = game_scalar.set_index("appid")
game_features = game_features.join(genre_pivot, how="left")
game_features = game_features.join(cat_pivot, how="left")
game_features = game_features.join(plat_pivot, how="left")
game_features = game_features.join(game_publisher.set_index("appid"), how="left")

# Fill NaN in multi-hot columns with 0 (games missing from junction tables)
game_features = game_features.fillna(0)

print(f"Static game feature matrix: {game_features.shape[0]:,} games x {game_features.shape[1]} features")
print(f"Missing values: {game_features.isnull().sum().sum()}")
game_features.head()

Static game feature matrix: 32,883 games x 94 features
Missing values: 0


,is_free,required_age,metacritic_score,mat_final_price_log,genre_Accounting,genre_Action,genre_Adventure,genre_Animation & Modeling,genre_Audio Production,genre_Casual,...,cat_Touch Only Option,cat_Tracked Controller Support,cat_VR Only,cat_VR Support,cat_VR Supported,cat_Valve Anti-Cheat enabled,platform_linux,platform_mac,platform_windows,publisher_portfolio_size
appid,,,,,,,,,,,,,,,,,,,,,
400,0,0,90.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,5.0
1300,0,0,75.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1313,0,0,77.0,6.907755,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,36.0
1520,0,0,84.0,7.090077,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0
1640,0,0,84.0,6.396930,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,76.0


## 3. Leakage-Safe LOO Aggregate Features

For each (user, game) row, we compute summary statistics about the user's behavior
and the game's reception. These features must be computed carefully to avoid
**label leakage** — where the feature inadvertently contains the label we're predicting.

### What features are computed

**User-level** (what kind of reviewer is this user?):

| Feature | Meaning |
|---|---|
| `user_positive_ratio` | What % of this user's reviews are positive? |
| `user_avg_playtime` | How much does this user typically play a game? |
| `user_price_preference` | What price range does this user typically buy? |
| `user_review_count_loo` | How many reviews has this user written? |

**Game-level** (how is this game received?):

| Feature | Meaning |
|---|---|
| `game_positive_ratio` | What % of this game's reviews are positive? |
| `game_avg_playtime` | How much do people typically play this game? |
| `game_review_count_loo` | How many reviews does this game have? |

**Cross-level**:

| Feature | Meaning |
|---|---|
| `playtime_vs_user_avg` | Does this user play this game more or less than their average? |
| `developer_avg_rating` | How well-received are this developer's other games? |

### Why Leave-One-Out is needed

The target label is `voted_up` (did the user recommend this game?). If the feature
computation includes the very label we're predicting, the model gets to peek at the
answer.

**Without LOO (leaky):** Say user A has 4 reviews — games X, Y, Z all positive, game W
negative. If we compute `user_positive_ratio = 3/4 = 75%` and use that for all of
user A's rows including game W, the feature is contaminated — game W's negative vote is
already baked into that 75%. Worse, `game_positive_ratio` for game W would include user
A's own vote, handing the model the answer disguised as a feature.

**With LOO (leakage-safe):** When computing features for (user A, game W), we exclude
that interaction:

```
user_positive_ratio for (A, W) = (3 positive from X,Y,Z) / (3 reviews) = 100%
                                  ↑ game W's vote is NOT counted

game_positive_ratio for (A, W) = (votes from OTHER users on W) / (count excluding A)
                                  ↑ user A's vote is NOT counted
```

No feature for row (A, W) contains any information from the label of row (A, W).

### Efficient implementation: the subtract-one trick

Instead of recomputing aggregates from scratch for every row, we:

1. Pre-compute totals per user/game from all training data
2. For each row, subtract that row's contribution:

```python
user_positive_ratio = (user_voted_up_sum - this_row_voted_up) / (user_review_count - 1)
```

Same result as "recompute without this row" but O(1) per row instead of O(n).

### Train vs. Test vs. Negatives — why the rules differ

| Dataset | LOO needed? | Why |
|---|---|---|
| **Train rows** | Yes — subtract current (u, g) | The row's own `voted_up` and `playtime` are in the totals; must exclude |
| **Test rows** | No exclusion | Test interactions were never in the totals (computed from train only) |
| **Synthetic negatives** | No exclusion | No real interaction exists, so there's nothing in the totals to leak |

### Developer-level LOO

Same logic, one level up. `developer_avg_rating` is the average `game_positive_ratio`
across a developer's games. When predicting for game W by developer D:
- **Train**: exclude game W's ratio from developer D's average
- **Test/negatives**: use the full developer average

### 3a. User-Level LOO Aggregates

In [ ]:
# Precompute user totals from training data
user_totals = train_df.groupby("author_steamid").agg(
    user_voted_up_sum=("voted_up", "sum"),
    user_review_count=("voted_up", "count"),
    user_playtime_sum=("author_playtime_forever", "sum"),
).astype(float)

print(f"User totals computed for {len(user_totals):,} users")
user_totals.head()

User totals computed for 36,161 users


,user_voted_up_sum,user_review_count,user_playtime_sum
author_steamid,,,
76561197960268765,1.0,2.0,884.0
76561197960272177,2.0,2.0,1534.0
76561197960272191,3.0,6.0,104.0
76561197960273880,17.0,17.0,18085.0
76561197960275132,4.0,6.0,590.0


### 3b. User Price Preference Totals

Pre-compute per-user price sum and count for leakage-safe LOO computation of
`user_price_preference` — the average price of games a user typically reviews.

In [ ]:
# Map each game to its price (raw USD cents); fall back to inverse log1p if raw price unavailable
if "mat_final_price" in apps.columns:
    game_price_map = apps.set_index("appid")["mat_final_price"].fillna(0).astype(float)
else:
    game_price_map = (np.expm1(apps.set_index("appid")["mat_final_price_log"].fillna(0))).astype(float)

train_df_price = train_df[["author_steamid", "appid"]].copy()
train_df_price["mat_final_price"] = train_df_price["appid"].map(game_price_map).fillna(0)

user_price_totals = train_df_price.groupby("author_steamid").agg(
    user_price_sum=("mat_final_price", "sum"),
    user_price_count=("mat_final_price", "count"),
).astype(float)

# Join price totals into user_totals
user_totals = user_totals.join(user_price_totals, how="left")
user_totals[["user_price_sum", "user_price_count"]] = user_totals[["user_price_sum", "user_price_count"]].fillna(0)

print(f"User price totals added: {len(user_price_totals):,} users")
print(f"User totals columns: {list(user_totals.columns)}")

User price totals added: 36,161 users
User totals columns: ['user_voted_up_sum', 'user_review_count', 'user_playtime_sum', 'user_price_sum', 'user_price_count']


### 3c. Game-Level LOO Aggregates

In [ ]:
# --- Game-level LOO aggregates (on training data) ---

game_totals = train_df.groupby("appid").agg(
    game_voted_up_sum=("voted_up", "sum"),
    game_review_count=("voted_up", "count"),
    game_playtime_sum=("author_playtime_forever", "sum"),
).astype(float)

print(f"Game totals computed for {len(game_totals):,} games")
game_totals.head()

Game totals computed for 31,551 games


,game_voted_up_sum,game_review_count,game_playtime_sum
appid,,,
400,16.0,17.0,19258.0
1300,2.0,2.0,306.0
1313,0.0,1.0,110.0
1520,1.0,1.0,298.0
1640,2.0,2.0,1397.0


### 3d. Developer-Level Aggregates

In [ ]:
# --- Developer-level aggregates (on training data) ---

# First compute game_positive_ratio (global, not LOO) for developer aggregation
game_pos_ratio_global = (game_totals["game_voted_up_sum"] / game_totals["game_review_count"]).rename("game_pos_ratio")

# Map game -> developer
game_dev = app_developers.drop_duplicates(subset="appid", keep="first").set_index("appid")["developer_id"]

# Compute developer avg rating across their games (from training data)
dev_ratings = game_pos_ratio_global.to_frame().join(game_dev, how="left")
dev_avg = dev_ratings.groupby("developer_id")["game_pos_ratio"].agg(["sum", "count"]).rename(
    columns={"sum": "dev_rating_sum", "count": "dev_game_count"}
)

print(f"Developer aggregates computed for {len(dev_avg):,} developers")
dev_avg.head()

Developer aggregates computed for 20,343 developers


,dev_rating_sum,dev_game_count
developer_id,,
9.0,0.714286,1
16.0,0.600000,1
18.0,0.833333,1
20.0,1.000000,1
22.0,0.800000,1


### 3e. LOO Feature Computation Function

Assembles all LOO features for a given DataFrame of (user, game) rows.
For training rows, excludes the current interaction; for test/negative rows,
uses full training aggregates.

In [ ]:
def compute_loo_features(df, user_totals, game_totals, dev_avg, game_dev, is_train=True):
    """
    Compute leave-one-out aggregate features for each (user, game) row.

    For training rows: exclude the current (u,g) interaction from aggregates.
    For test rows: use full training aggregates (no exclusion needed since
                   the test interaction was never in the training totals).
    For synthetic negatives: use full training aggregates (no real interaction to exclude).
    """
    result = df[["author_steamid", "appid"]].copy()

    # Merge user totals
    result = result.merge(user_totals, left_on="author_steamid", right_index=True, how="left")
    # Merge game totals
    result = result.merge(game_totals, left_on="appid", right_index=True, how="left")

    # Look up the price of the current game for LOO price computation
    current_price = df["appid"].map(game_price_map).fillna(0).astype(float).values

    if is_train:
        # LOO: subtract current interaction's contribution
        voted_up_float = df["voted_up"].astype(float).values
        playtime_float = df["author_playtime_forever"].astype(float).values

        # User LOO
        result["user_positive_ratio"] = (
            (result["user_voted_up_sum"] - voted_up_float) /
            (result["user_review_count"] - 1).clip(lower=1)
        )
        result["user_avg_playtime"] = (
            (result["user_playtime_sum"] - playtime_float) /
            (result["user_review_count"] - 1).clip(lower=1)
        )
        result["user_review_count_loo"] = result["user_review_count"] - 1

        # user_price_preference (LOO): exclude current game's price
        result["user_price_preference"] = (
            (result["user_price_sum"] - current_price) /
            (result["user_price_count"] - 1).clip(lower=1)
        )

        # Game LOO
        result["game_positive_ratio"] = (
            (result["game_voted_up_sum"] - voted_up_float) /
            (result["game_review_count"] - 1).clip(lower=1)
        )
        result["game_avg_playtime"] = (
            (result["game_playtime_sum"] - playtime_float) /
            (result["game_review_count"] - 1).clip(lower=1)
        )
        result["game_review_count_loo"] = result["game_review_count"] - 1
    else:
        # Test / negative: use full training aggregates (no exclusion)
        result["user_positive_ratio"] = result["user_voted_up_sum"] / result["user_review_count"].clip(lower=1)
        result["user_avg_playtime"] = result["user_playtime_sum"] / result["user_review_count"].clip(lower=1)
        result["user_review_count_loo"] = result["user_review_count"]
        result["user_price_preference"] = result["user_price_sum"] / result["user_price_count"].clip(lower=1)

        result["game_positive_ratio"] = result["game_voted_up_sum"] / result["game_review_count"].clip(lower=1)
        result["game_avg_playtime"] = result["game_playtime_sum"] / result["game_review_count"].clip(lower=1)
        result["game_review_count_loo"] = result["game_review_count"]

    # Playtime vs user avg
    result["playtime_vs_user_avg"] = (
        df["author_playtime_forever"].values / result["user_avg_playtime"].clip(lower=1).values
    )

    # Developer avg rating (LOO: exclude current game from developer's average)
    result = result.merge(game_dev.rename("developer_id"), left_on="appid", right_index=True, how="left")
    result = result.merge(dev_avg, left_on="developer_id", right_index=True, how="left")

    if is_train:
        # Exclude current game's positive_ratio from developer average
        current_game_ratio = result["game_voted_up_sum"] / result["game_review_count"].clip(lower=1)
        result["developer_avg_rating"] = (
            (result["dev_rating_sum"] - current_game_ratio) /
            (result["dev_game_count"] - 1).clip(lower=1)
        )
    else:
        result["developer_avg_rating"] = result["dev_rating_sum"] / result["dev_game_count"].clip(lower=1)

    # Keep only the engineered features
    feature_cols = [
        "user_positive_ratio", "user_avg_playtime", "user_review_count_loo", "user_price_preference",
        "game_positive_ratio", "game_avg_playtime", "game_review_count_loo",
        "playtime_vs_user_avg", "developer_avg_rating",
    ]
    return result[feature_cols].fillna(0)

print("LOO feature function defined.")

LOO feature function defined.


### 3f. Compute LOO Features for Train and Test

In [ ]:
# Compute LOO features for train and test
train_loo = compute_loo_features(train_df, user_totals, game_totals, dev_avg, game_dev, is_train=True)
test_loo = compute_loo_features(test_df, user_totals, game_totals, dev_avg, game_dev, is_train=False)

print(f"Train LOO features: {train_loo.shape}")
print(f"Test LOO features:  {test_loo.shape}")
print(f"\nSample train LOO features:")
train_loo.head()

Train LOO features: (173832, 9)
Test LOO features:  (51063, 9)

Sample train LOO features:


,user_positive_ratio,user_avg_playtime,user_review_count_loo,user_price_preference,game_positive_ratio,game_avg_playtime,game_review_count_loo,playtime_vs_user_avg,developer_avg_rating
0,1.0,842.0,1.0,999.0,0.666667,45.166667,6.0,0.049881,0.000000
1,0.0,42.0,1.0,799.0,1.000000,307.666667,15.0,20.047619,0.000000
3,1.0,1032.0,1.0,1499.0,0.666667,265.555556,9.0,0.486434,0.571429
4,1.0,502.0,1.0,999.0,1.000000,193.666667,3.0,2.055777,0.000000
6,0.6,14.8,5.0,539.0,0.600000,210.300000,10.0,2.027027,0.000000


## 4. Domain-Informed Negative Sampling

### Why negative sampling is needed

The dataset contains only reviewed (user, game) pairs — explicit interactions where a
user chose to write a review. With ~6 reviews per user across ~33K games (99.98%
sparsity), the model has no signal for "user u would NOT like game g" without synthetic
negatives. These negatives teach the model to distinguish genuinely appealing games from
the vast majority a user would not engage with.

### Why random sampling fails for game recommendations

A naïve approach — sampling uniformly from all non-reviewed games — produces **trivially
easy negatives**. A hardcore strategy gamer is unlikely to review a casual farming
simulator, but that tells the model nothing useful. The model learns to separate
obviously unrelated games rather than making fine-grained distinctions within genres the
user actually cares about.

### Domain-specific sampling strategy

We exploit three properties of how players interact with games on Steam:

**1. Genre-proportional sampling (not just genre overlap)**

Binary genre overlap is insufficient: the top genres — Indie (23K games), Action (14K),
Adventure (14K), Casual (14K) — collectively cover ~92% of all games in the dataset.
A user who plays *any* combination of these genres would have nearly every game as a
candidate, making the filter useless.

Instead, we sample genres **proportional to the user's actual play intensity**. If a
user has reviewed 5 Action games but only 1 Casual game, ~83% of their negatives come
from Action and ~17% from Casual. This produces harder negatives concentrated in genres
where the user is most active — games they plausibly encountered but chose not to
review.

**2. Platform compatibility filtering**

Steam games are platform-specific: a game may support Windows, Mac, Linux, or any
combination. A user who has only ever reviewed Mac-compatible games likely does not have
access to Windows-exclusive titles. Including those as negatives adds noise — the user
didn't skip these games by choice; they simply couldn't play them.

We infer each user's available platforms from their review history and reject any
sampled game that shares no platform with the user.

**3. Popularity-weighted sampling within genres**

Within each genre, we weight sampling by game popularity (review count). A popular game
with thousands of reviews that a user *didn't* review is a much stronger negative signal
than an obscure title with 2 reviews that the user likely never heard of. Popular skips
indicate deliberate disinterest; obscure skips indicate lack of exposure.

### Sampling method: two-stage rejection sampling

For each positive training interaction, we generate `NEG_RATIO` (4) negatives:

```
For each negative slot:
  1. Sample a genre from the user's normalised genre distribution
  2. Sample a game from that genre (popularity-weighted, O(1) via searchsorted)
  3. Reject if:
     (a) game was already reviewed by this user, or
     (b) game is not available on any platform the user has used
  4. Retry up to 5 times if rejected
```

This is much faster than the alternative of building a per-user candidate pool and
calling `np.random.choice(replace=False, p=weights)` over ~30K candidates (O(n) per
user × 36K users). The rejection rate is <1%, so nearly all slots fill on the first
attempt.

### Ratio and labelling

- **4 negatives per positive**: standard ratio for implicit feedback systems (He et al., 2017)
- **Label**: `voted_up = False` for synthetic negatives (same as real negative reviews)
- **Flag**: `is_synthetic_negative = True` to distinguish from real thumbs-down reviews
- **Deduplication**: (user, game) pairs are deduplicated after sampling

In [ ]:
NEG_RATIO = 4  # 4 negatives per positive

# ── Pre-compute per-genre candidate pools ────────────────────────────
# For each genre: sorted game array + cumulative popularity weights
# (enables O(1) weighted sampling via searchsorted)
all_train_games = np.sort(train_df["appid"].unique())
game_popularity = train_df.groupby("appid").size().reindex(all_train_games, fill_value=1)

unique_genres = sorted(app_genres["genre_name"].unique())
genre_games = {}      # genre -> np.array of appids
genre_cum_weights = {} # genre -> cumulative normalised popularity weights

for genre in unique_genres:
    games = app_genres[app_genres["genre_name"] == genre]["appid"].unique()
    games = games[np.isin(games, all_train_games)]
    if len(games) == 0:
        continue
    weights = np.array([game_popularity.get(g, 1) for g in games], dtype=np.float64)
    cum_w = np.cumsum(weights)
    cum_w /= cum_w[-1]
    genre_games[genre] = games
    genre_cum_weights[genre] = cum_w

# ── Pre-compute user profiles ────────────────────────────────────────
# User genre distribution: normalised count of reviews per genre
# (captures taste intensity, not just binary overlap)
user_genre_df = (
    train_df[["author_steamid", "appid"]]
    .merge(app_genres[["appid", "genre_name"]], on="appid")
    .groupby(["author_steamid", "genre_name"]).size()
    .rename("w").reset_index()
)
user_genre_dists = {}  # uid -> (genre_names_array, cumulative_probs)
for uid, grp in user_genre_df.groupby("author_steamid"):
    genres = grp["genre_name"].values
    weights = grp["w"].values.astype(np.float64)
    cum_w = np.cumsum(weights)
    cum_w /= cum_w[-1]
    user_genre_dists[uid] = (genres, cum_w)

# User platform set (inferred from games they've reviewed)
user_platform_sets = (
    train_df[["author_steamid", "appid"]]
    .merge(app_platforms[["appid", "platform_id"]], on="appid")
    [["author_steamid", "platform_id"]].drop_duplicates()
    .groupby("author_steamid")["platform_id"].apply(frozenset).to_dict()
)

# Game platform set
game_platform_sets = app_platforms.groupby("appid")["platform_id"].apply(frozenset).to_dict()

# User reviewed games
user_reviewed_sets = train_df.groupby("author_steamid")["appid"].apply(frozenset).to_dict()

print(f"Genres with candidates: {len(genre_games)}")
print(f"Users with genre profiles: {len(user_genre_dists):,}")
print(f"Users with platform info: {len(user_platform_sets):,}")

# ── Two-stage rejection sampling ─────────────────────────────────────
# For each positive training row, generate NEG_RATIO negatives.
# Stage 1: sample genre from user's genre distribution
# Stage 2: sample game from that genre, weighted by popularity
# Reject if: (a) game was reviewed by user, or (b) platform-incompatible

rng = np.random.default_rng(42)

source_uids = train_df["author_steamid"].values
n_neg_total = len(source_uids) * NEG_RATIO
neg_uids = np.repeat(source_uids, NEG_RATIO)
neg_appids = np.zeros(n_neg_total, dtype=all_train_games.dtype)
filled = np.zeros(n_neg_total, dtype=bool)

MAX_ATTEMPTS = 5
for attempt in range(MAX_ATTEMPTS):
    todo = np.where(~filled)[0]
    if len(todo) == 0:
        break

    for idx in todo:
        uid = neg_uids[idx]
        dist = user_genre_dists.get(uid)
        if dist is None:
            continue
        genres_arr, cum_probs = dist

        # Stage 1: pick genre
        gi = min(np.searchsorted(cum_probs, rng.random()), len(genres_arr) - 1)
        genre = genres_arr[gi]
        if genre not in genre_games:
            continue

        # Stage 2: pick game from genre (popularity-weighted)
        g_arr = genre_games[genre]
        g_cum = genre_cum_weights[genre]
        game_i = min(np.searchsorted(g_cum, rng.random()), len(g_arr) - 1)
        game = g_arr[game_i]

        # Reject if reviewed
        if game in user_reviewed_sets.get(uid, frozenset()):
            continue

        # Reject if platform-incompatible
        u_plats = user_platform_sets.get(uid, frozenset())
        g_plats = game_platform_sets.get(game, frozenset())
        if u_plats and g_plats and not (u_plats & g_plats):
            continue

        neg_appids[idx] = game
        filled[idx] = True

    print(f"  Attempt {attempt + 1}: filled {filled.sum():,} / {n_neg_total:,} ({filled.mean():.2%})")

# ── Assemble result and deduplicate ──────────────────────────────────
neg_df = pd.DataFrame({
    "author_steamid": neg_uids[filled],
    "appid": neg_appids[filled],
})
# Drop duplicate (user, game) pairs — rejection sampling can produce repeats
neg_df = neg_df.drop_duplicates(subset=["author_steamid", "appid"])

neg_df["voted_up"] = False
neg_df["is_synthetic_negative"] = True
neg_df["author_playtime_forever"] = 0.0

print(f"\nGenerated {len(neg_df):,} synthetic negatives (after dedup)")
print(f"Ratio: {len(neg_df) / len(train_df):.1f} negatives per real interaction")
print(f"Unique users with negatives: {neg_df['author_steamid'].nunique():,}")
print(f"Unique games sampled: {neg_df['appid'].nunique():,}")

Genres with candidates: 28
Users with genre profiles: 36,150
Users with platform info: 36,161
  Attempt 1: filled 692,798 / 695,328 (99.64%)
  Attempt 2: filled 695,218 / 695,328 (99.98%)
  Attempt 3: filled 695,253 / 695,328 (99.99%)
  Attempt 4: filled 695,254 / 695,328 (99.99%)
  Attempt 5: filled 695,254 / 695,328 (99.99%)

Generated 690,537 synthetic negatives (after dedup)
Ratio: 4.0 negatives per real interaction
Unique users with negatives: 36,150
Unique games sampled: 31,079


In [ ]:
# Compute LOO features for negative samples (no exclusion needed)
neg_loo = compute_loo_features(neg_df, user_totals, game_totals, dev_avg, game_dev, is_train=False)

print(f"Negative sample LOO features: {neg_loo.shape}")
neg_loo.head()

Negative sample LOO features: (690537, 9)


,user_positive_ratio,user_avg_playtime,user_review_count_loo,user_price_preference,game_positive_ratio,game_avg_playtime,game_review_count_loo,playtime_vs_user_avg,developer_avg_rating
0,0.5,442.0,2.0,899.0,0.000000,21.500000,4.0,0.0,0.413239
1,0.5,442.0,2.0,899.0,1.000000,54.000000,8.0,0.0,0.886440
2,0.5,442.0,2.0,899.0,0.727273,261.727273,11.0,0.0,0.833917
3,0.5,442.0,2.0,899.0,0.666667,2096.000000,3.0,0.0,0.666667
4,0.5,442.0,2.0,899.0,0.333333,120.000000,3.0,0.0,0.333333


## 5. BERT Embeddings of `short_description`

Safe to pre-compute: `short_description` is editorial content, independent of user experience.

In [1]:
import os
import pandas as pd
import joblib

embedding_path = os.path.join(OUTPUT_DIR, "game_description_embeddings.parquet")

if os.path.exists(embedding_path):
    print(f"Loading cached embeddings from {embedding_path}")
    desc_emb_df = pd.read_parquet(embedding_path)
    print(f"Loaded embeddings: {desc_emb_df.shape}")
else:
    from sentence_transformers import SentenceTransformer
    from sklearn.decomposition import PCA

    # Load model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Prepare descriptions
    desc_df = apps[["appid", "short_description"]].copy()
    desc_df["short_description"] = desc_df["short_description"].fillna("").astype(str)

    # Encode
    print(f"Encoding {len(desc_df):,} descriptions...")
    embeddings_384 = model.encode(
        desc_df["short_description"].tolist(),
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"Raw embeddings: {embeddings_384.shape}")

    # PCA reduce to 128 dims
    pca = PCA(n_components=128, random_state=42)
    embeddings_128 = pca.fit_transform(embeddings_384)
    print(f"PCA-reduced embeddings: {embeddings_128.shape}")
    print(f"Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
    joblib.dump(pca, "game_desc_pca.pkl")

    # Store as DataFrame
    desc_emb_cols = [f"desc_emb_{i}" for i in range(128)]
    desc_emb_df = pd.DataFrame(embeddings_128, columns=desc_emb_cols, index=desc_df["appid"])

desc_emb_df.head()


NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
# from sentence_transformers import SentenceTransformer
# from sklearn.decomposition import PCA

# # Load model
# model = SentenceTransformer("all-MiniLM-L6-v2")

# # Prepare descriptions
# desc_df = apps[["appid", "short_description"]].copy()
# desc_df["short_description"] = desc_df["short_description"].fillna("").astype(str)

# # Encode
# print(f"Encoding {len(desc_df):,} descriptions...")
# embeddings_384 = model.encode(
#     desc_df["short_description"].tolist(),
#     batch_size=256,
#     show_progress_bar=True,
#     normalize_embeddings=True,
# )
# print(f"Raw embeddings: {embeddings_384.shape}")

# # PCA reduce to 128 dims
# pca = PCA(n_components=128, random_state=42)
# embeddings_128 = pca.fit_transform(embeddings_384)
# print(f"PCA-reduced embeddings: {embeddings_128.shape}")
# print(f"Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# # Store as DataFrame
# desc_emb_cols = [f"desc_emb_{i}" for i in range(128)]
# desc_emb_df = pd.DataFrame(embeddings_128, columns=desc_emb_cols, index=desc_df["appid"])
# desc_emb_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 32,883 descriptions...


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

Raw embeddings: (32883, 384)
PCA-reduced embeddings: (32883, 128)
Variance explained: 78.2%


,desc_emb_0,desc_emb_1,desc_emb_2,desc_emb_3,desc_emb_4,desc_emb_5,desc_emb_6,desc_emb_7,desc_emb_8,desc_emb_9,...,desc_emb_118,desc_emb_119,desc_emb_120,desc_emb_121,desc_emb_122,desc_emb_123,desc_emb_124,desc_emb_125,desc_emb_126,desc_emb_127
appid,,,,,,,,,,,,,,,,,,,,,
400,0.127334,0.234454,-0.045320,0.093814,-0.060112,0.101858,-0.007598,0.076641,0.199637,0.011200,...,-0.057913,-0.013363,0.024660,-0.008150,0.044710,0.019605,0.006236,0.006766,0.029913,-0.006292
1300,-0.086652,-0.266915,0.037698,0.010966,-0.101552,0.004867,0.140006,0.115462,-0.047205,0.029091,...,-0.023922,-0.066933,0.078879,0.011939,-0.004152,-0.063366,-0.086202,-0.039160,-0.006131,-0.008258
1313,-0.104068,-0.056749,0.135682,-0.094452,-0.048875,0.034507,-0.059555,0.056446,-0.020930,-0.129082,...,0.022062,0.018692,0.004197,0.002764,0.008816,-0.084858,-0.020806,-0.010951,0.092830,-0.002066
1520,0.202559,0.001342,-0.223662,0.068996,-0.229597,0.005506,0.216479,0.033829,-0.045168,0.028776,...,-0.003956,-0.023499,0.117325,0.035725,0.018913,-0.002188,0.109917,-0.000737,-0.020540,0.060800
1640,-0.036931,-0.078148,-0.158223,-0.169378,-0.118772,0.170498,0.060643,0.142795,0.003209,-0.102198,...,-0.006402,-0.100778,-0.068321,-0.082844,-0.017296,0.000266,0.010155,-0.016985,0.115138,-0.097554


## 6. User Genre Preference Vector (Leakage-Safe)

For each user, compute a genre preference profile by aggregating the genre multi-hot
vectors of their **positively reviewed** games, weighted by playtime.

This captures which genres a user gravitates toward and how much time they invest in each.

**Derivation**: For each user u in training data:
1. Select only positively reviewed games (voted_up=True)
2. For each such game, get its genre multi-hot vector
3. Weight each genre vector by the user's playtime on that game
4. Sum the weighted vectors and normalize by total playtime

The result is a continuous vector (one dimension per genre) representing the user's
genre preference distribution. Stored as a separate user-level feature matrix.

> **Note**: User taste profiles from BERT-encoded review text are computed in Phase 2,
> where they are used for content-based similarity scoring.

In [ ]:
# --- user_genre_vector: playtime-weighted genre profile from positively reviewed games ---

# Use genre_pivot (already computed in Section 2a) — columns are genre_{name}, index is appid
genre_cols = list(genre_pivot.columns)
n_genres = len(genre_cols)

# Filter training data to positive reviews only
train_positive = train_df[train_df["voted_up"] == True][["author_steamid", "appid", "author_playtime_forever"]].copy()

# Merge genre vectors for each game
train_pos_genres = train_positive.merge(
    genre_pivot, left_on="appid", right_index=True, how="left"
)

# Fill NaN genres with 0 (games without genre info)
train_pos_genres[genre_cols] = train_pos_genres[genre_cols].fillna(0)

# Weight each genre column by playtime
playtime_vals = train_pos_genres["author_playtime_forever"].values.reshape(-1, 1)
weighted_genres = train_pos_genres[genre_cols].values * playtime_vals

# Create weighted DataFrame for aggregation
weighted_df = pd.DataFrame(weighted_genres, columns=genre_cols, index=train_pos_genres.index)
weighted_df["author_steamid"] = train_pos_genres["author_steamid"].values
weighted_df["playtime"] = train_pos_genres["author_playtime_forever"].values

# Sum weighted genre vectors per user
user_genre_sum = weighted_df.groupby("author_steamid")[genre_cols].sum()
user_playtime_total = weighted_df.groupby("author_steamid")["playtime"].sum()

# Normalize by total playtime (avoid division by zero)
user_genre_vector = user_genre_sum.div(user_playtime_total.clip(lower=1), axis=0)

# Rename columns to user_genre_* for clarity
user_genre_vector.columns = [f"user_{c}" for c in genre_cols]  # e.g., user_genre_Action

print(f"User genre vectors: {user_genre_vector.shape[0]:,} users x {user_genre_vector.shape[1]} genres")
print(f"Users with positive reviews: {len(user_genre_vector):,} / {train_df['author_steamid'].nunique():,}")
print(f"\nSample user genre vector (top genres for first user):")
sample = user_genre_vector.iloc[0].sort_values(ascending=False)
print(sample[sample > 0].head(10).to_string())
user_genre_vector.head()

User genre vectors: 32,907 users x 28 genres
Users with positive reviews: 32,907 / 36,161

Sample user genre vector (top genres for first user):
user_genre_Action        1.0
user_genre_Adventure     1.0
user_genre_RPG           1.0
user_genre_Indie         1.0
user_genre_Simulation    1.0
user_genre_Strategy      1.0


,user_genre_Accounting,user_genre_Action,user_genre_Adventure,user_genre_Animation & Modeling,user_genre_Audio Production,user_genre_Casual,user_genre_Design & Illustration,user_genre_Early Access,user_genre_Education,user_genre_Free To Play,...,user_genre_Racing,user_genre_Sexual Content,user_genre_Simulation,user_genre_Software Training,user_genre_Sports,user_genre_Strategy,user_genre_Utilities,user_genre_Video Production,user_genre_Violent,user_genre_Web Publishing
author_steamid,,,,,,,,,,,,,,,,,,,,,
76561197960268765,0.0,1.000000,1.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,...,0.000000,0.0,1.000000,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
76561197960272177,0.0,0.327249,0.327249,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
76561197960272191,0.0,1.000000,0.405405,0.0,0.0,0.243243,0.0,0.243243,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
76561197960273880,0.0,0.137517,0.440807,0.0,0.0,0.728394,0.0,0.180592,0.0,0.0,...,0.155433,0.0,0.231407,0.0,0.005087,0.005087,0.0,0.0,0.0,0.0
76561197960275132,0.0,0.315534,0.456311,0.0,0.0,0.635922,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.315534,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0


## 7. Save All Pre-Computed Features

In [ ]:
import os

# Add split and is_synthetic_negative columns for downstream use
train_df["is_synthetic_negative"] = False
test_df["is_synthetic_negative"] = False

# Concatenate LOO features with their respective DataFrames
train_with_loo = pd.concat([train_df.reset_index(drop=True), train_loo.reset_index(drop=True)], axis=1)
test_with_loo = pd.concat([test_df.reset_index(drop=True), test_loo.reset_index(drop=True)], axis=1)
neg_with_loo = pd.concat([neg_df.reset_index(drop=True), neg_loo.reset_index(drop=True)], axis=1)

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save everything
train_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "train_with_features.parquet"), index=False)
test_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "test_with_features.parquet"), index=False)
neg_with_loo.to_parquet(os.path.join(OUTPUT_DIR, "negatives_with_features.parquet"), index=False)
game_features.to_parquet(os.path.join(OUTPUT_DIR, "game_features_static.parquet"))
desc_emb_df.to_parquet(os.path.join(OUTPUT_DIR, "game_description_embeddings.parquet"))
user_genre_vector.to_parquet(os.path.join(OUTPUT_DIR, "user_genre_vectors.parquet"))

print(f"All pre-computed features saved to {OUTPUT_DIR}/")
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:<45} {size_mb:>8.1f} MB")

All pre-computed features saved to /content/drive/MyDrive/feature_engineered_data/

  game_description_embeddings.parquet               24.8 MB
  game_features_static.parquet                       0.5 MB
  negatives_with_features.parquet                    5.9 MB
  test_with_features.parquet                        25.2 MB
  train_with_features.parquet                       79.1 MB
  user_genre_vectors.parquet                         1.4 MB


## 8. Summary — What's Ready for Each Phase

### Output files

| File | Contents | Used by |
|---|---|---|
| `train_with_features.parquet` | Training reviews + all LOO aggregate features | Phase 1, 3 |
| `test_with_features.parquet` | Test reviews + all LOO aggregate features | Phase 1, 3 |
| `negatives_with_features.parquet` | Synthetic negatives + aggregate features | Phase 1, 3 |
| `game_features_static.parquet` | Genre/category/platform multi-hot + scalar features | Phase 2, 3 |
| `game_description_embeddings.parquet` | BERT embeddings of `short_description` (128-dim) | Phase 2, 3 |
| `user_genre_vectors.parquet` | Playtime-weighted genre preference vectors (28-dim) | Phase 3 |

### Engineered features by level

**User-level features** (one value per user per row, LOO-safe):

| Feature | Description | Derivation |
|---|---|---|
| `user_positive_ratio` | Fraction of positive reviews | LOO: (sum voted_up - this row) / (count - 1) |
| `user_avg_playtime` | Mean playtime across reviewed games | LOO: (sum playtime - this row) / (count - 1) |
| `user_price_preference` | Average price of reviewed games | LOO: (sum price - this game's price) / (count - 1) |
| `user_review_count_loo` | Number of reviews written | count - 1 (excl. current row) |

**Game-level features** (one value per game per row, LOO-safe):

| Feature | Description | Derivation |
|---|---|---|
| `game_positive_ratio` | Fraction of positive reviews for this game | LOO: (sum voted_up - this row) / (count - 1) |
| `game_avg_playtime` | Mean playtime across all reviewers of this game | LOO: (sum playtime - this row) / (count - 1) |
| `game_review_count_loo` | Number of reviews for this game | count - 1 (excl. current row) |

**Cross-level features**:

| Feature | Description | Derivation |
|---|---|---|
| `playtime_vs_user_avg` | User's playtime on this game / their avg playtime | Ratio of per-row playtime to LOO user avg |
| `developer_avg_rating` | Avg positive ratio across this developer's games | LOO: excludes current game from developer avg |

**Static game features** (in `game_features_static.parquet`):

| Feature | Dims | Description |
|---|---|---|
| Genre multi-hot (`genre_*`) | 28 | One binary column per English genre |
| Category multi-hot (`cat_*`) | 58 | One binary column per category |
| Platform multi-hot (`platform_*`) | 3 | windows / mac / linux |
| `is_free` | 1 | Binary |
| `required_age` | 1 | Minimum age rating |
| `metacritic_score` | 1 | Genre-median imputed |
| `mat_final_price_log` | 1 | Log-transformed price |
| `publisher_portfolio_size` | 1 | Number of games by this publisher |

**User profile features**:

| Feature | Dims | File | Description |
|---|---|---|---|
| User genre vector | 28 | `user_genre_vectors.parquet` | Playtime-weighted genre distribution from positively reviewed games |

**Game embedding features**:

| Feature | Dims | File | Description |
|---|---|---|---|
| Description embedding | 128 | `game_description_embeddings.parquet` | PCA-reduced `all-MiniLM-L6-v2` encoding of `short_description` |

### Negative sampling

- **~695K synthetic negatives** (4:1 ratio) generated via domain-informed two-stage rejection sampling
- Genre-proportional, platform-compatible, popularity-weighted
- LOO features computed with full training aggregates (no exclusion needed)

### Computed in later phases

| Feature | Produced by | Consumed by |
|---|---|---|
| User taste profiles from review text (128-dim) | Phase 2 | Phase 2 |
| NCF user/item embeddings (32-dim) | Phase 1 | Phase 3 |
| NCF predicted scores | Phase 1 | Phase 4 |
| Content-based similarity scores | Phase 2 | Phase 4 |

In [ ]:
# Final verification: shapes and dtypes
print("=" * 60)
print("PRE-COMPUTED FEATURE SUMMARY")
print("=" * 60)
print(f"\nTrain set:     {train_with_loo.shape} (includes {train_with_loo['is_synthetic_negative'].sum()} synthetic negatives)")
print(f"Test set:      {test_with_loo.shape}")
print(f"Negatives:     {neg_with_loo.shape}")
print(f"Game features: {game_features.shape}")
print(f"BERT desc:     {desc_emb_df.shape}")
print(f"User genre:    {user_genre_vector.shape}")
print(f"\nLOO feature columns: {list(train_loo.columns)}")
print(f"Static game feature columns ({game_features.shape[1]} total): {list(game_features.columns[:10])} ... + {game_features.shape[1]-10} more")

PRE-COMPUTED FEATURE SUMMARY

Train set:     (173832, 36) (includes 0 synthetic negatives)
Test set:      (51063, 36)
Negatives:     (690537, 14)
Game features: (32883, 94)
BERT desc:     (32883, 128)
User genre:    (32907, 28)

LOO feature columns: ['user_positive_ratio', 'user_avg_playtime', 'user_review_count_loo', 'user_price_preference', 'game_positive_ratio', 'game_avg_playtime', 'game_review_count_loo', 'playtime_vs_user_avg', 'developer_avg_rating']
Static game feature columns (94 total): ['is_free', 'required_age', 'metacritic_score', 'mat_final_price_log', 'genre_Accounting', 'genre_Action', 'genre_Adventure', 'genre_Animation & Modeling', 'genre_Audio Production', 'genre_Casual'] ... + 84 more
